In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from joblib import dump, load
import os
import yaml

from helpers.geometry_map import make_geometry_map, build_dist_tree_from_map
from helpers.data_transforms import load_in_data, local_phi_transformation

from tqdm import tqdm
from helpers.material_map import apply_material_map_hybrid, build_masked_datasets
from helpers.flow import build_xy_z_lookup, snap_z_to_detector_xy, build_z_lookup, snap_z_to_detector
#plt.style.use("../science.mplstyle")

In [2]:
with open("configs.yaml", "r") as f:
    configs = yaml.safe_load(f)

PATH_TO_OUTPUT_DIR = configs["PATH_TO_OUTPUT_DIR"]
PATH_TO_DATA_DIR = configs["PATH_TO_DATA_DIR"]
ALL_COLLECTIONS = configs["ALL_COLLECTIONS"]
FEATURE_ORDER = configs["FEATURE_ORDER"]
FEATURE_INDICES_DICT = configs["FEATURE_INDICES_DICT"]

First we build the lookup tree that allows us to assign cell IDs to the ML samples.

In [ ]:
cache_path = f"{PATH_TO_OUTPUT_DIR}/geometry_tree.joblib"

if os.path.exists(cache_path):
    print("Loading cached geometry tree...")
    data = load(cache_path)
    tree_map = data["tree_map"]
    cellid_layer_map = data["cellid_layer_map"]
else:



    geo_map_path = f"{PATH_TO_OUTPUT_DIR}/geometry_map.txt"

    path_to_data = f"{PATH_TO_DATA_DIR}/nuGun_pT_0_50_reco_0.h5"
    df = pd.read_hdf(path_to_data, key="df") 
    make_geometry_map(
        df,
        ALL_COLLECTIONS,
        geo_map_path,
        num_events_per_col=50_000_000,
        
)
    print("Building geometry tree...")
    tree_map, cellid_layer_map = build_dist_tree_from_map(geo_map_path)

    dump(
        {
            "tree_map": tree_map,
            "cellid_layer_map": cellid_layer_map,
        },
        cache_path,
        compress=3,
    )

The cell below load in the npy flow samples. 

In [ ]:


NUM_COND_INPUTS = 4
BASIS = "rphi"
log_vars = []

paths_to_samples = {
    "OuterTrackerBarrelCollection": "/scratch/midway3/rmastand/muon_collider/zuko_outputs/OTBC_cond4/",
    "OuterTrackerEndcapCollection": "/scratch/midway3/rmastand/muon_collider/zuko_outputs/OTEC_cond4/",
    "InnerTrackerBarrelCollection": "/scratch/midway3/rmastand/muon_collider/zuko_outputs/ITBC_cond4/",
    "InnerTrackerEndcapCollection": "/scratch/midway3/rmastand/muon_collider/zuko_outputs/ITEC_cond4/",
    "VertexBarrelCollection": "/scratch/midway3/rmastand/muon_collider/zuko_outputs/VBC_cond4/",
    "VertexEndcapCollection": "/scratch/midway3/rmastand/muon_collider/zuko_outputs/VEC_cond4/",
                    }


In [ ]:
# load in samples

all_data_dir, all_samples_dir = {}, {}

for col_name in ALL_COLLECTIONS:

    small_id = ''.join([c for c in col_name if c.isupper()])
    X, feature_labels = load_in_data([col_name], BASIS, PATH_TO_DATA_DIR, 1, NUM_COND_INPUTS, FEATURE_ORDER)
    
    all_data_dir[col_name] = X

   
    all_samples_dir[col_name] = np.load(f"{paths_to_samples[col_name]}/generated_samples.npy")


For the flow samples, we need to "snap" the endcap z coordinate to the sensitive region of the detector.

In [ ]:
for col_name in ALL_COLLECTIONS:

    if "Endcap" in col_name:
        plt.figure()
    
        z_idx = FEATURE_INDICES_DICT["z"]
        side_idx = FEATURE_INDICES_DICT["side"]
        layer_idx = FEATURE_INDICES_DICT["layer"]
    
        # original samples
        z_samples = all_samples_dir[col_name][:, z_idx]
        sides = all_samples_dir[col_name][:, side_idx]
        layers = all_samples_dir[col_name][:, layer_idx]
    
    

        z_lookup = build_xy_z_lookup(
            all_data_dir[col_name],
            FEATURE_INDICES_DICT["side"],
            FEATURE_INDICES_DICT["layer"],
            FEATURE_INDICES_DICT["r"],
            FEATURE_INDICES_DICT["phi"],
            FEATURE_INDICES_DICT["z"],
        )


        plt.hist(
            all_data_dir[col_name][:, z_idx],
            bins=np.linspace(-2000, 2000, 1000),
            histtype="step",
            label="Sim BIB"
        )

        plt.hist(
            all_samples_dir[col_name][:,FEATURE_INDICES_DICT["z"]],
            bins=np.linspace(-2000, 2000, 1000),
            histtype="step",
            label="ML BIB (before snapping)"
        )

            

        z_samples_snapped =  snap_z_to_detector_xy(
            all_samples_dir[col_name][:, FEATURE_INDICES_DICT["r"]],
            all_samples_dir[col_name][:, FEATURE_INDICES_DICT["phi"]],
            sides,
            layers,
            z_lookup,
        )

        all_samples_dir[col_name][:,FEATURE_INDICES_DICT["z"]] = z_samples_snapped
    
        # plot truth vs snapped samples
        
    
        plt.hist(
            z_samples_snapped,
            bins=np.linspace(-2000, 2000, 1000),
            histtype="step",
            label="ML BIB (after snapping)"
        )

        
        plt.title(col_name)
        plt.yscale("log")
        plt.legend()
        plt.xlabel("$z$ [mm]")
        plt.ylabel("Counts")
        plt.legend(loc = (1,0))
        plt.show()



In [ ]:
for col_name in ALL_COLLECTIONS:
    print(col_name)
    print(all_data_dir[col_name].shape, all_samples_dir[col_name].shape)

    fig, ax = plt.subplots(1, 7, figsize = (20, 5))
    for i in range(7):
        ax[i].hist(all_data_dir[col_name][:,i], histtype = "step", bins = 100, density = True)
        ax[i].hist(all_samples_dir[col_name][:,i], histtype = "step", bins = 100, density = True)
        ax[i].set_xlabel(f"feature {i}")
    ax[-1].legend()
    plt.show()

Now we actually assign the cell IDs to the ML BIB by sending them through the lookup tree.

In [ ]:



def get_col_hits_per_layer(data):

    

    col_hits_per_layer = {col: {} for col in ALL_COLLECTIONS}
    dists  = {col: {} for col in ALL_COLLECTIONS}
    cell_ids = {col: {} for col in ALL_COLLECTIONS}
    valid_masks = {col: {} for col in ALL_COLLECTIONS}

    
    
    for col_name in ALL_COLLECTIONS:
    
    
        # -----------------------------
        # 1. Extract coordinates
        # -----------------------------
        r = data[col_name][:,FEATURE_INDICES_DICT[col_name]["r"]]
        phi = data[col_name][:,FEATURE_INDICES_DICT[col_name]["phi"]]
        z = data[col_name][:,FEATURE_INDICES_DICT[col_name]["z"]]
        x = r*np.cos(phi)
        y = r*np.sin(phi)
        
        points = np.stack([x,y,z], axis=1)   # shape (N, 3)
    
        # -----------------------------
        # 2. Mask out NaNs / infs
        # -----------------------------
        valid_mask = np.isfinite(points).all(axis=1)
        points_valid = points[valid_mask]
        valid_masks[col_name] = valid_mask
    
        # Skip if nothing valid
        if len(points_valid) == 0:
            continue
    
        # -----------------------------
        # 3. Batch KDTree query
        # -----------------------------
        dists_batch, idx_batch = tree_map[col_name].query(points_valid)
    
        # -----------------------------
        # 4. Remove invalid KDTree hits
        # -----------------------------
        valid_idx_mask = idx_batch < len(cellid_layer_map[col_name])
    
        idx_batch = idx_batch[valid_idx_mask]
        dists[col_name]  = dists_batch[valid_idx_mask]
    
        # -----------------------------
        # 5. Map to (cell_id, layer)
        # -----------------------------
        cell_layer = cellid_layer_map[col_name][idx_batch]
        layers = cell_layer[:, 1]
        cell_ids[col_name] = cell_layer[:, 0]
    
        # -----------------------------
        # 6. Count hits per layer (vectorized)
        # -----------------------------
        unique_layers, counts = np.unique(layers, return_counts=True)
    
        for layer, count in zip(unique_layers, counts):
            col_hits_per_layer[col_name][layer] = (
                col_hits_per_layer[col_name].get(layer, 0) + count
            )


    return col_hits_per_layer, dists, cell_ids, valid_masks

First apply masks to the collections, then assign cell ids

In [ ]:
col_hits_per_layer_truth, dists_truth, cell_ids_truth, valid_mask = get_col_hits_per_layer(all_data_dir)



flow_samples_masked, flow_samples_masked_stratified = build_masked_datasets(all_data_dir, all_samples_dir, ALL_COLLECTIONS, NUM_COND_INPUTS, FEATURE_INDICES_DICT, stratify=False)
col_hits_per_layer_samples, dists_samples, cell_ids_samples, valid_masks = get_col_hits_per_layer(flow_samples_masked)


Diagnostic plot

In [ ]:
for key in all_data_dir.keys():
    print(key, len(all_data_dir[key]))
    print(key, len(flow_samples_masked[key]))


plt.figure()
for i, col in enumerate(ALL_COLLECTIONS):
    plt.hist(dists_truth[col], bins = 100, histtype = "step", density = True, linestyle = "dashed", color = f"C{i}")
plt.yscale("log")
plt.xlabel("Distance from KDtree [mm]")
plt.ylabel("Density")
plt.legend()
plt.show()

plt.figure()
for i, col in enumerate(ALL_COLLECTIONS):
    plt.hist(dists_samples[col], bins = np.linspace(0, 15, 100), histtype = "step", density = True, color = f"C{i}", label = col)
    num_pass = dists_samples[col] < 1
    print(col,100*sum(num_pass)/len(num_pass))
plt.yscale("log")
plt.xlabel("Distance from KDtree [mm]")
plt.ylabel("Density")
plt.legend(loc = (1,0))
plt.show()

More sanity checks (not important)

In [ ]:
collections_order = [
    'VertexBarrelCollection',
    'VertexEndcapCollection',
    'InnerTrackerBarrelCollection',
    'InnerTrackerEndcapCollection',
    'OuterTrackerBarrelCollection',
    'OuterTrackerEndcapCollection'
]


def make_plot(col_hits_per_layer):
    
    # Prepare x positions
    width = 0.1  # width of each bar
    x_offsets = np.arange(0, len(col_hits_per_layer[collections_order[0]]))  # base positions for first collection
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Keep track of total x positions
    x_pos = 0
    x_labels = []
    x_ticks = []
    
    for coll in collections_order:
        layers = sorted(col_hits_per_layer[coll].keys())  # sort layers numerically
        counts = [col_hits_per_layer[coll][l] for l in layers]
    
        # compute positions for this collection
        positions = x_pos + np.arange(len(layers))
        ax.bar(positions, counts, width=0.8, label=coll)
        
        # collect labels and ticks
        x_labels.extend([f"{l}" for l in layers])
        x_ticks.extend(positions)
        
        # update x_pos for next collection to avoid overlap
        x_pos = positions[-1] + 1  # add gap between collections
    
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels, rotation=45, ha='right')
    ax.set_ylabel("Number of hits")
    ax.set_title("Hits per Detector Layer")
    ax.legend()
    plt.tight_layout()
    plt.yscale("log")
    plt.savefig("figures/hits_per_detector_layer.png")
    plt.show()


make_plot(col_hits_per_layer_truth)
make_plot(col_hits_per_layer_samples)

Randomly save out 10% of the ML BIB samples so that digitization and reconstruction run faster

In [ ]:
hits_dict_inside_bounds = {
    "OuterTrackerBarrelCollection":3_280_678,
    "OuterTrackerEndcapCollection":1_465_825,
    "InnerTrackerBarrelCollection":3_926_196,
    "InnerTrackerEndcapCollection":1_614_679,
    "VertexBarrelCollection":2_129_166,
    "VertexEndcapCollection":4_096_371,
}

hits_dict_no_condition = {
    "OuterTrackerBarrelCollection":6_787_500,
    "OuterTrackerEndcapCollection":2_825_825,
    "InnerTrackerBarrelCollection":6_639_439,
    "InnerTrackerEndcapCollection":2_360_215,
    "VertexBarrelCollection":2_641_857,
    "VertexEndcapCollection":5_325_807,
}

for col in collections_order:

    print(col)
    #print(flow_samples_masked[col].shape)
    #print(cell_ids_samples[col].shape)


    loc = np.concatenate([flow_samples_masked[col], cell_ids_samples[col].reshape(-1,1)], axis = 1)

    
    #print(loc.shape)

    n_samples = hits_dict_no_condition[col]

    idx = np.random.choice(len(loc), size=n_samples, replace=False)

    loc_subset = loc[idx]

    print("flow samples", loc_subset.shape)
    np.save(f"/pscratch/sd/r/rmastand/muon_collider/npys/flow_samples/{col}_23_06_local_phi_no_bound_condition.npy", loc_subset)

    #print(loc.shape,)

In [ ]:
for col_name in collections:


    flow1 = np.load(f"/pscratch/sd/r/rmastand/muon_collider/npys/flow_samples/{col_name}_23_06_global_phi_no_bound_condition.npy")
    flow2 = np.load(f"/pscratch/sd/r/rmastand/muon_collider/npys/flow_samples/{col_name}_23_06_local_phi_no_bound_condition.npy")
    train = np.load(f"/pscratch/sd/r/rmastand/muon_collider/npys/flow_samples/{col_name}_23_06_data_no_bound_condition.npy")
    print(col_name)
    print(flow1.shape, flow2.shape)
    print(train.shape)

    fig, ax = plt.subplots(1, 8, figsize = (20, 5))
    for i in range(8):
        ax[i].hist(train[:,i], histtype = "step", bins = 100, density = True)
        ax[i].hist(flow1[:,i], histtype = "step", bins = 100, density = True)
        ax[i].hist(flow2[:,i], histtype = "step", bins = 100, density = True)
        
        ax[i].set_xlabel(f"feature {i}")
    ax[-1].legend()
    plt.show()